# Phase 1 & 2: Explorative Datenanalyse (EDA)

**Ziel:** Das Dataset kennenlernen, Klassenverteilung verstehen, erste visuelle Erkenntnisse gewinnen.

**StackFuel-Themen:** Native Python · EDA mit Pandas & Matplotlib · Statistik

In [ ]:
import sys
sys.path.append('..')  # src.utils importierbar machen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image

from src.utils import (
    get_class_dirs,
    count_images_per_class,
    show_sample_images,
    plot_class_distribution,
    load_images_as_array,
    get_pixel_stats
)

print('Setup OK')

## 1.1 Dataset-Struktur erkunden

In [ ]:
# Ordnerstruktur anzeigen
data_dir = Path('../data/raw')

for item in sorted(data_dir.rglob('*')):
    depth = len(item.relative_to(data_dir).parts)
    if depth <= 2:  # nur 2 Ebenen tief
        print('  ' * (depth - 1) + item.name)

## 1.2 Klassenverteilung

In [ ]:
# Anzahl Bilder pro Klasse als DataFrame
counts_train = count_images_per_class('train')
df_counts = pd.DataFrame(list(counts_train.items()), columns=['klasse', 'anzahl'])
df_counts['anteil_%'] = (df_counts['anzahl'] / df_counts['anzahl'].sum() * 100).round(1)

print(df_counts.to_string(index=False))

In [ ]:
# Visualisierung der Verteilung
plot_class_distribution('train')

**Beobachtung:**  
_Hier deine eigene Erkenntnis eintragen: Ist das Dataset ausgeglichen? Gibt es seltene Klassen?_

## 1.3 Beispielbilder anzeigen

Das `fig, ax`-Muster von Matplotlib – wichtig laut StackFuel-Dokument!

In [ ]:
show_sample_images(split='train', n_per_class=4)

## 2.1 Pixelstatistiken pro Klasse

In [ ]:
class_dirs = get_class_dirs('train')
stats_list = []

for class_name, class_path in class_dirs.items():
    arr = load_images_as_array(class_path, size=(64, 64), max_images=100)
    stats = get_pixel_stats(arr)
    stats['klasse'] = class_name
    stats['n_bilder'] = len(arr)
    stats_list.append(stats)

df_stats = pd.DataFrame(stats_list).set_index('klasse')
df_stats.round(3)

## 2.2 RGB-Histogramme vergleichen

In [ ]:
fig, axes = plt.subplots(1, len(class_dirs), figsize=(5 * len(class_dirs), 3), sharey=True)

colors = ['red', 'green', 'blue']

for ax, (class_name, class_path) in zip(axes, class_dirs.items()):
    arr = load_images_as_array(class_path, size=(64, 64), max_images=50)
    for i, color in enumerate(colors):
        ax.hist(arr[:, :, :, i].flatten(), bins=50, alpha=0.5, color=color, label=color.upper())
    ax.set_title(class_name, fontsize=10)
    ax.set_xlabel('Pixelwert')

axes[0].set_ylabel('Häufigkeit')
axes[0].legend(fontsize=8)
fig.suptitle('RGB-Verteilung pro Klasse', fontsize=12)
plt.tight_layout()
plt.show()

## 2.3 Helligkeit im Vergleich (Boxplot)

In [ ]:
brightness_data = []

for class_name, class_path in class_dirs.items():
    arr = load_images_as_array(class_path, size=(64, 64), max_images=100)
    per_image_brightness = arr.mean(axis=(1, 2, 3))  # Mittelwert jedes Bildes
    for val in per_image_brightness:
        brightness_data.append({'klasse': class_name, 'helligkeit': val})

df_brightness = pd.DataFrame(brightness_data)

fig, ax = plt.subplots(figsize=(8, 4))
df_brightness.boxplot(column='helligkeit', by='klasse', ax=ax)
ax.set_title('Mittlere Bildhelligkeit pro Klasse')
ax.set_xlabel('Klasse')
ax.set_ylabel('Mittlere Helligkeit (0–1)')
plt.suptitle('')  # Standard-Titel von boxplot entfernen
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 2.4 Zusammenfassung der EDA

_Hier deine eigenen Erkenntnisse eintragen:_

- **Klassenverteilung:** ...
- **Visuelle Unterschiede zwischen den Klassen:** ...
- **RGB-Charakteristik:** ...
- **Hypothesen für die Modellierung:** ...